## Concept 10 — Self-RAG

### What is it?
In all previous concepts, the **pipeline decides what to do** and the LLM just answers.
Self-RAG flips this: the **LLM itself makes every decision** in the process.

The LLM makes 4 key decisions:
```
Decision 1 — Should I retrieve at all?
  → Maybe it already knows the answer from training

Decision 2 — Is each retrieved doc actually relevant?
  → Filter out irrelevant docs before using them

Decision 3 — Is my answer grounded in the docs?
  → Verify answer is supported by retrieved content, not hallucinated

Decision 4 — Is my answer complete and useful?
  → Self-assess quality before returning to user
```

### Why it's the most accurate
Every other RAG always retrieves, always uses all docs, always returns an answer.
Self-RAG skips retrieval when not needed, filters bad docs, and verifies its own answer.
Result: higher accuracy, fewer hallucinations, more efficient.

### Limitation (what Concept 11 fixes)
Self-RAG still does a single retrieval per question — cannot chain multiple lookups.
→ Concept 11 (Multi-hop RAG) handles complex questions requiring multiple retrieval steps.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries

chunks, vectorstore, embeddings, llm, prompt = setup()
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### Decision 1 — Should I Retrieve?
LLM decides if it needs external docs or already knows the answer.

In [ ]:
def should_retrieve(query):
    decision_prompt = (
        f"To answer this question, do you need to search external documents?\n"
        f"Or do you already know the answer from general knowledge?\n\n"
        f"Question: {query}\n\n"
        f"Answer YES (need external docs) or NO (already know):"
    )
    response = llm.invoke(decision_prompt).content.strip().upper()
    return "YES" in response

# Test
print(f"'What is 2+2?' → need retrieval: {should_retrieve('What is 2+2?')}")
print(f"'What is LangSmith persistence?' → need retrieval: {should_retrieve(TEST_QUERIES[0])}")

### Decision 2 — Is This Doc Relevant?
LLM filters out docs that won't help answer the question.

In [ ]:
def is_doc_relevant(query, doc):
    rel_prompt = (
        f"Is this document useful for answering the question?\n\n"
        f"Question: {query}\n"
        f"Document: {doc.page_content[:500]}\n\n"
        f"Answer YES or NO only:"
    )
    return "YES" in llm.invoke(rel_prompt).content.strip().upper()

def filter_relevant_docs(query, docs):
    relevant = [doc for doc in docs if is_doc_relevant(query, doc)]
    print(f"  Filtered: {len(relevant)}/{len(docs)} docs are relevant")
    return relevant

### Decision 3 — Is My Answer Grounded?
LLM checks whether its answer is actually supported by the retrieved docs.

In [ ]:
def is_answer_grounded(answer, context):
    ground_prompt = (
        f"Is this answer fully supported by the context provided?\n"
        f"Check that every claim in the answer appears in the context.\n\n"
        f"Context: {context[:800]}\n\n"
        f"Answer: {answer}\n\n"
        f"Reply YES (fully grounded) or NO (contains unsupported claims):"
    )
    return "YES" in llm.invoke(ground_prompt).content.strip().upper()

### Full Self-RAG Flow
All 4 decisions working together.

In [ ]:
def self_rag(query):
    print(f"Query: {query}")

    # Decision 1: Should I retrieve?
    need_retrieval = should_retrieve(query)
    print(f"  Decision 1 — Retrieve? {need_retrieval}")

    if not need_retrieval:
        answer = llm.invoke(query).content
        print("  Answered from training knowledge (no retrieval)")
        return answer

    # Retrieve
    docs = retriever.invoke(query)

    # Decision 2: Filter relevant docs
    relevant_docs = filter_relevant_docs(query, docs)

    if not relevant_docs:
        print("  No relevant docs found after filtering")
        return "I don't know — no relevant documents found in the knowledge base."

    context = format_docs(relevant_docs)

    # Generate answer
    answer = llm.invoke(prompt.format_messages(context=context, question=query)).content

    # Decision 3: Is the answer grounded?
    grounded = is_answer_grounded(answer, context)
    print(f"  Decision 3 — Answer grounded? {grounded}")

    if not grounded:
        print("  Regenerating answer (previous was not grounded)...")
        answer = llm.invoke(prompt.format_messages(context=context, question=query)).content

    return answer

### Test — Standard Queries

In [ ]:
run_queries(self_rag, label="Concept 10 — Self-RAG")

### Bonus — Question That Doesn't Need Retrieval
Watch Self-RAG skip retrieval entirely for a general knowledge question.

In [ ]:
print(self_rag("What is Python programming language?"))